<a href="https://colab.research.google.com/github/legna7816/ml-projects/blob/main/nlp_sentiment/nlp_step02_movie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_files
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [17]:
# 1. 데이터 불러오기 - IMDB 영화리뷰
# sklearn 내장 데이터셋 사용 (긍정/부정 리뷰 각 1000개씩)
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

from sklearn.datasets import fetch_20newsgroups

# 더 간단한 내장 데이터: 영화리뷰 느낌의 실습용 데이터 직접 만들기
reviews = [
    # 긍정 리뷰 (label=1)
    "this movie was absolutely fantastic and amazing",
    "great film loved every moment of it",
    "brilliant acting and wonderful story",
    "best movie i have ever seen highly recommend",
    "incredible cinematography and beautiful music",
    "outstanding performance by all the actors",
    "a masterpiece of modern cinema loved it",
    "funny heartwarming and deeply moving film",
    "superb direction and excellent screenplay",
    "one of the greatest movies of all time",

    # 부정 리뷰 (label=0)
    "this movie was terrible waste of time",
    "awful boring and completely unwatchable",
    "worst film i have ever seen absolutely horrible",
    "bad acting poor story and dull direction",
    "completely disappointing and frustrating experience",
    "nothing works in this dreadful movie",
    "tedious and painfully boring throughout",
    "ridiculous plot with terrible performances",
    "an absolute disaster avoid at all costs",
    "poorly made and utterly forgettable film"
]
labels = [1]*10 + [0]*10    # 1: 긍정, 0: 부정

In [18]:
# 2. 전처리 - TF-IDF로 텍스트 벡터화
# stop_words = 'english': the/is/a 같은 의미없는 단어 자동 제거
# max_feature=500: 가장 중요한 단어 500개만 사용 (큰 데이터에서 차원 제한)
tfidf = TfidfVectorizer(stop_words='english', max_features=500)
X = tfidf.fit_transform(reviews)    # 텍스트 -> 숫자 벡터 (TF-IDF)
y = np.array(labels)

print('벡터 크기: ', X.shape)  # (문장 수, 단어 수)
print('사용된 단어 목록: ', tfidf.get_feature_names_out())

벡터 크기:  (20, 65)
사용된 단어 목록:  ['absolute' 'absolutely' 'acting' 'actors' 'amazing' 'avoid' 'awful' 'bad'
 'beautiful' 'best' 'boring' 'brilliant' 'cinema' 'cinematography'
 'completely' 'costs' 'deeply' 'direction' 'disappointing' 'disaster'
 'dreadful' 'dull' 'excellent' 'experience' 'fantastic' 'film'
 'forgettable' 'frustrating' 'funny' 'great' 'greatest' 'heartwarming'
 'highly' 'horrible' 'incredible' 'loved' 'masterpiece' 'modern' 'moment'
 'movie' 'movies' 'moving' 'music' 'outstanding' 'painfully' 'performance'
 'performances' 'plot' 'poor' 'poorly' 'recommend' 'ridiculous'
 'screenplay' 'seen' 'story' 'superb' 'tedious' 'terrible' 'time'
 'unwatchable' 'utterly' 'waste' 'wonderful' 'works' 'worst']


In [19]:
# 3. 모델 학습 - 타이타닉의 fit/predict 패턴과 동일
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# 모델 1: LogisticRegression
log_model = LogisticRegression()
log_model.fit(X_train, y_train)
log_pred = log_model.predict(X_test)
print('Logistic 정확도: ', accuracy_score(y_test, log_pred))

# 모델 2: MultinomialNB(나이브베이즈) - nlp에서 자주 쓰는 모델
# "이 단어들이 나왔을 때, 긍정일 확률 vs 부정일 확률"을 계산함
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)
print('NaiveBayes 정확도: ', accuracy_score(y_test, nb_pred))
print(classification_report(y_test, nb_pred, target_names=['부정', '긍정']))

Logistic 정확도:  0.3333333333333333
NaiveBayes 정확도:  0.3333333333333333
              precision    recall  f1-score   support

          부정       0.33      1.00      0.50         2
          긍정       0.00      0.00      0.00         4

    accuracy                           0.33         6
   macro avg       0.17      0.50      0.25         6
weighted avg       0.11      0.33      0.17         6



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [20]:
# 4. 새로운 리뷰 예측하기
# tfidf는 학습 때 본 단어로만 벡터화하므로 transform만 사용 (fit_transform 아님)
new_reviews = [
    "this film was absolutely wonderful and fantastic",
    "terrible movie boring and awful acting"
]
X_new = tfidf.transform(new_reviews)    # fit은 이미 위에서 진행함 -> transform만 진행
print('예측결과 (1: 긍정, 0: 부정): ', log_model.predict(X_new))

예측결과 (1: 긍정, 0: 부정):  [0 0]


In [24]:
# 5. 직접 해보기 (TODO)
# 5-1. 교차검증(cross_val_score)으로 LogisticRegression vs NaiveBayes 평균 정확도 비교
log_cv = cross_val_score(LogisticRegression(max_iter=200), X, y, cv=5)
nb_cv = cross_val_score(MultinomialNB(), X, y, cv=5)

print('Logistic 평균: ', log_cv.mean())
print('NaiveBayes 평균: ', nb_cv.mean())

# TODO 2: 아래 new_review2로 nb_model도 예측해서 두 모델 결과 비교
new_review2 = ["incredible story with great acting and beautiful direction"]
X_new2 = tfidf.transform(new_reviews)
print('\n예측결과: ', log_model.predict(X_new2))

#  TODO 3: tfidf에서 fit_transform 대신 transform만 써야 하는 이유를 한 줄로 설명해보기

# -> fit_tranform을 새 데이터에 쓰면 학습 때 만든 단어 목록과 가중치가 덮어씌워져서,
#    모델이 학습한 기준과 달라지기 때문임.

Logistic 평균:  0.45
NaiveBayes 평균:  0.5

예측결과:  [0 0]
